<a id="object-detection"></a>
# VideoDB Understanding: Object Detection

Detect objects on Sandbox Compute and draw normalized bounding boxes over frames generated by VideoDB.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/object-detection/object-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install and connect

Object detection requires an active compatible small sandbox. This notebook lists your active sandboxes, creates one if needed, and stops it at the end — sandboxes accrue charges while they run.

In [ ]:
!pip install -q videodb python-dotenv pillow requests

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import SandboxTier, connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Connected to VideoDB")
print(f"Collection: {collection.id}")

## 2. Choose a video

In [ ]:
VIDEO_URL = os.getenv(
    "VIDEODB_VIDEO_URL",
    "https://www.youtube.com/watch?v=vVlEVRKv4is",  # Silicon Valley clip: Gilfoyle is free for hire
)

video = collection.upload(url=VIDEO_URL)

# Already have this video in your account? Comment out the upload above
# and reference it by its VideoDB video ID instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()

## 3. Create a small sandbox

Object detection uses a small-tier sandbox. List your active sandboxes first — reuse one with `conn.get_sandbox("sb-...")` if a compatible one is already running, or create a new one below and wait until it is active.

In [ ]:
active_sandboxes = conn.list_sandboxes(status="active")

print(f"Active sandboxes: {len(active_sandboxes)}")
for sb in active_sandboxes:
    print(f"{sb.id} | {sb.name} | {sb.tier} | {sb.status}")

In [ ]:
sandbox = conn.create_sandbox(tier=SandboxTier.small)
print("Sandbox created")
print(f"ID: {sandbox.id}")
print(f"Status: {sandbox.status}")

### Wait until the sandbox is active

Run this cell again if the sandbox is still provisioning. It reuses the sandbox created above instead of creating another one.


In [ ]:
sandbox.wait_for_ready(timeout=300, interval=5)
print("Sandbox ready")
print(f"ID: {sandbox.id}")
print(f"Status: {sandbox.status}")

<a id="configuration"></a>
## 4. Configure detections

- `sandbox_id` selects the active sandbox that will run object detection.
- `labels` limits detections to useful classes.
- `confidence_threshold` removes weak detections.
- Interval sampling controls temporal coverage. The default samples every three seconds to keep this 100-second demonstration bounded; decrease it when recall matters more than latency.
- `include_bounding_boxes` keeps geometry in the output.

In [ ]:
OBJECT_LABELS = ["person", "chair", "laptop", "cell phone", "book", "bottle"]

understanding = video.understand(
    analyzers=[{
        "type": "object_detection",
        "name": "objects",
        "sampling": {"strategy": "interval", "every": 3},
        "config": {
            # Object detection is dispatched to this active sandbox.
            "sandbox_id": sandbox.id,
            "labels": OBJECT_LABELS,
            "confidence_threshold": 0.35,
            "include_bounding_boxes": True,
        },
    }],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding submitted")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")

In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Understanding complete")
print(f"Status: {understanding.status}")

## 5. Inspect detections

Each frame contains `timestamp_sec` and a list of detections. Each detection has `label`, `score`, `box`, and `box_format="xyxy_normalized"`.

In [ ]:
objects_output = understanding.get_analyzer("objects").get_output()
detection_frames = [
    frame
    for scene in objects_output.get("scenes", [])
    for frame in (scene.get("data") or {}).get("frames", [])
]

print("Detection summary:")
print(f"- Scenes: {len(objects_output.get('scenes', []))}")
print(f"- Sampled frames: {len(detection_frames)}")
print(f"- Detections: {sum(len(frame.get('detections', [])) for frame in detection_frames)}")

<a id="bounding-boxes"></a>
## 6. Draw bounding boxes

VideoDB generates the source frame at each detection timestamp. Normalized coordinates are scaled to that frame's width and height.

In [ ]:
from io import BytesIO

import requests
from IPython.display import display
from PIL import Image as PILImage
from PIL import ImageDraw


def draw_boxes(frame):
    timestamp = max(float(frame.get("timestamp_sec") or 0), 0.001)
    asset = video.generate_thumbnail(time=timestamp)
    image_url = asset.url or asset.generate_url()
    response = requests.get(image_url, timeout=60)
    response.raise_for_status()

    image = PILImage.open(BytesIO(response.content)).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size

    for detection in frame.get("detections", []):
        box = detection.get("box")
        if not box or len(box) != 4:
            continue
        x1, y1, x2, y2 = box
        rectangle = (x1 * width, y1 * height, x2 * width, y2 * height)
        label = f"{detection.get('label', 'object')} {detection.get('score', 0):.2f}"
        draw.rectangle(rectangle, outline="#00ff88", width=max(2, width // 300))
        label_box = draw.textbbox((rectangle[0], rectangle[1]), label)
        draw.rectangle(label_box, fill="#00ff88")
        draw.text((rectangle[0], rectangle[1]), label, fill="black")

    print(f"Frame at {timestamp:.2f}s — {len(frame.get('detections', []))} detections")
    display(image)


frames_with_objects = [frame for frame in detection_frames if frame.get("detections")]
if not frames_with_objects:
    raise RuntimeError("No detections found in the sampled frames")

for frame in frames_with_objects[:3]:
    draw_boxes(frame)

## 7. Use the output

Index `frames.detections.label` for filtering and facets, and `frames.detections.score` for confidence filters or sorting. Object detections can also be an input to a downstream VLM in the [pipeline guide](../multi-analyzer-pipelines.ipynb).

## 8. Cleanup

Remove the Understanding run only once its artifact is no longer needed. Stopping the sandbox ends compute billing.

In [ ]:
DELETE_RUN = False
STOP_SANDBOX = True

if DELETE_RUN:
    understanding.delete()
    print("Deleted", understanding.id)

if STOP_SANDBOX:
    sandbox.stop()
    print("Stopping", sandbox.id)